# Fidelity vs. Sparse Budget (Fig. 3 / Tables VI–VII)

Reproduces the BASS-vs-fixed-basis fidelity and participation-ratio sweep at
`N = 20` across the three circuit families used in Sec. VI.B of the
manuscript: 1D brickwork, 2D brickwork, and RFIM. The RFIM disorder
convention is longitudinal, `W = 2`, matching Sec. VI.A(d) exactly (the
earlier `sigma_h/h = 0.1` transverse-disorder label used in an older draft
has been fully retired from the codebase and does not appear anywhere below).

**What changed from the previous version of this notebook:** significance
testing no longer lives in an ad-hoc, per-notebook code block. It goes
through the shared, reviewer-checked helpers in
`src/experimentation/runner.py`:

- `paired_comparison_stats` — median [IQR], geometric-mean ratio with a
  4000-resample bootstrap 95% CI, Wilson-score win-rate CI, a one-sided
  paired Wilcoxon signed-rank test, and a matched-pairs rank-biserial
  effect size (Kerby, 2014) for every single (family, k) cell.
- `build_significance_table` — assembles all of the above into one table
  with a Holm-Bonferroni **family-wise** correction across every (family, k)
  test performed here (not per-cell, which would understate the true
  false-positive risk).

The exact same two functions are used, unmodified, in
`MPS_Comparison.ipynb` and `SchmidtWeightedTruncation.ipynb`, so there is
exactly one implementation of "how we report a paired comparison" in the
whole repository.

Set `QUICK_TEST = True` below for a fast sanity-check run (small N, few
trials) before committing to the full `N_TRIALS = 100` publication run.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for c in (start, *start.parents):
        if (c / "src").is_dir() and (c / "requirements.txt").exists():
            return c
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "tests" / "results"
FIG_DIR = REPO_ROOT / "tests" / "figures"
DATA_DIR.mkdir(exist_ok=True, parents=True)
FIG_DIR.mkdir(exist_ok=True, parents=True)

from src.experimentation.runner import (
    run_trial,
    make_brickwork,
    make_brickwork_2d,
    make_rfim,
    trial_stats,
    paired_comparison_stats,
    build_significance_table,
    format_significance_table,
)
from src.visualization.style import mplstyle
import matplotlib.pyplot as plt

mplstyle()
print("Imports OK. Repo root:", REPO_ROOT)


In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
QUICK_TEST = False  # flip to True first to sanity-check the pipeline quickly

BASS_KW = dict(optimize_every=5, buffer_factor=1, use_2qubit_rotations=False)
SEED_BASE = 20260601
N_BOOT = 4000

if QUICK_TEST:
    N_TRIALS = 5
    K_VALUES = [50, 500, 5000]
    FAMILIES = {
        "1D Brickwork": {"N": 10, "gen": lambda Nq, rng: make_brickwork(Nq, depth=6, rng=rng)},
        "2D Brickwork": {"N": 6, "gen": lambda Nq, rng: make_brickwork_2d(2, 3, depth=4, rng=rng)},
        "RFIM (W=2)": {"N": 10, "gen": lambda Nq, rng: make_rfim(Nq, rng, layers=5, W=2.0)},
    }
else:
    N_TRIALS = 100
    K_VALUES = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]
    FAMILIES = {
        "1D Brickwork": {"N": 20, "gen": lambda Nq, rng: make_brickwork(Nq, depth=6, rng=rng)},
        "2D Brickwork": {"N": 20, "gen": lambda Nq, rng: make_brickwork_2d(4, 5, depth=4, rng=rng)},
        "RFIM (W=2)": {"N": 20, "gen": lambda Nq, rng: make_rfim(Nq, rng, layers=5, W=2.0)},
    }

NPZ_PATH = DATA_DIR / ("fidelity_pr_sweep_quick.npz" if QUICK_TEST else "fidelity_pr_sweep.npz")
META_PATH = NPZ_PATH.with_suffix(".meta.json")
EXPECTED_META = {
    "N_TRIALS": N_TRIALS,
    "K_VALUES": K_VALUES,
    "seed_base": SEED_BASE,
    "families": {name: spec["N"] for name, spec in FAMILIES.items()},
}
print(f"N_TRIALS={N_TRIALS}, k in {K_VALUES}")
print("Families:", {n: s["N"] for n, s in FAMILIES.items()})


In [ ]:
# ---------------------------------------------------------------------------
# Run (or load cached) sweep. One call to run_trial() per (family, k, trial)
# -- run_trial() already calls each simulator exactly once internally, so
# there is no redundant-simulate()-call risk here.
# ---------------------------------------------------------------------------
def run_or_load():
    if NPZ_PATH.exists() and META_PATH.exists():
        stored_meta = json.loads(META_PATH.read_text())
        if stored_meta == EXPECTED_META:
            print("Loaded from cache:", NPZ_PATH)
            return dict(np.load(NPZ_PATH, allow_pickle=True))
        print("Cache meta mismatch -- re-running.")

    out = {}
    fam_names = list(FAMILIES.keys())
    for fi, fam_name in enumerate(fam_names):
        spec = FAMILIES[fam_name]
        Nq, gen = spec["N"], spec["gen"]
        for ki, k in enumerate(K_VALUES):
            f_fixed = np.full(N_TRIALS, np.nan)
            f_bass = np.full(N_TRIALS, np.nan)
            pr_fixed = np.full(N_TRIALS, np.nan)
            pr_bass = np.full(N_TRIALS, np.nan)
            for t in range(N_TRIALS):
                seed = SEED_BASE + fi * 1_000_000 + ki * 10_000 + t
                ff, fb, prf, prb = run_trial(gen, Nq, k, seed=seed, bass_kwargs=BASS_KW)
                f_fixed[t], f_bass[t] = ff, fb
                pr_fixed[t], pr_bass[t] = prf, prb
            key = f"{fam_name}||{k}"
            out[f"{key}||f_fixed"] = f_fixed
            out[f"{key}||f_bass"] = f_bass
            out[f"{key}||pr_fixed"] = pr_fixed
            out[f"{key}||pr_bass"] = pr_bass
            gm_fixed = np.exp(np.mean(np.log(np.clip(f_fixed, 1e-300, None))))
            gm_bass = np.exp(np.mean(np.log(np.clip(f_bass, 1e-300, None))))
            print(
                f"  {fam_name:14s} k={k:6d}  GM(F_fixed)={gm_fixed:.3e}  "
                f"GM(F_bass)={gm_bass:.3e}  ratio={gm_bass/gm_fixed:.2f}x"
            )

    np.savez(NPZ_PATH, **out)
    META_PATH.write_text(json.dumps(EXPECTED_META, indent=2))
    print("Saved to", NPZ_PATH)
    return out


data = run_or_load()


In [ ]:
# ---------------------------------------------------------------------------
# Publication-grade significance table (replaces the old bare-geomean output)
# ---------------------------------------------------------------------------
records = []
for fam_name in FAMILIES:
    for k in K_VALUES:
        key = f"{fam_name}||{k}"
        f_fixed = data[f"{key}||f_fixed"]
        f_bass = data[f"{key}||f_bass"]
        records.append(
            paired_comparison_stats(
                f_bass, f_fixed,
                label=f"{fam_name}, k={k}",
                alternative="greater",  # H1: BASS stochastically dominates fixed-basis
                n_boot=N_BOOT,
            )
        )

sig_df, sig_meta = build_significance_table(records, alpha=0.05)
print(format_significance_table(sig_df, sig_meta))

csv_path = DATA_DIR / "fidelity_pr_significance_table.csv"
sig_df.to_csv(csv_path, index=False)
print("\nSaved:", csv_path)


In [ ]:
# ---------------------------------------------------------------------------
# Fig. 3-style plot: fidelity vs. sparse budget k, one panel per family
# ---------------------------------------------------------------------------
fam_names = list(FAMILIES.keys())
fig, axes = plt.subplots(1, len(fam_names), figsize=(5 * len(fam_names), 4), sharey=True)
if len(fam_names) == 1:
    axes = [axes]

for ax, fam_name in zip(axes, fam_names):
    gm_fixed, lo_fixed, hi_fixed = [], [], []
    gm_bass, lo_bass, hi_bass = [], [], []
    for k in K_VALUES:
        key = f"{fam_name}||{k}"
        sf = trial_stats(data[f"{key}||f_fixed"])
        sb = trial_stats(data[f"{key}||f_bass"])
        gm_fixed.append(sf["geomean"]); lo_fixed.append(sf["gm_ci_lo"]); hi_fixed.append(sf["gm_ci_hi"])
        gm_bass.append(sb["geomean"]); lo_bass.append(sb["gm_ci_lo"]); hi_bass.append(sb["gm_ci_hi"])

    ax.plot(K_VALUES, gm_fixed, "o--", label="Fixed-basis", color="0.4")
    ax.fill_between(K_VALUES, lo_fixed, hi_fixed, alpha=0.15, color="0.4")
    ax.plot(K_VALUES, gm_bass, "o-", label="BASS", color="#0C5DA5")
    ax.fill_between(K_VALUES, lo_bass, hi_bass, alpha=0.15, color="#0C5DA5")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Sparse budget $k$")
    ax.set_title(fam_name)

axes[0].set_ylabel("Fidelity $F$ (geometric mean, 95% CI)")
axes[0].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_fidelity_pr_sweep.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "fig_fidelity_pr_sweep.png", dpi=200, bbox_inches="tight")
plt.show()
print("Figure saved.")


**Manuscript note:** the numeric claims in Tables VI–VII should now cite
`sig_df` (i.e. `tests/results/fidelity_pr_significance_table.csv`), which
carries the Holm-corrected p-value and rank-biserial effect size for every
cell already reported there as a bare geometric-mean ratio -- no manuscript
text changes required beyond adding "(Wilcoxon p < ..., Holm-corrected;
effect size r = ...)" alongside each existing ratio, pulled directly from
this table.
